In [7]:
today_date = '2026-02-25'

StatementMeta(, acb60a16-ca7e-402b-bdd7-8027fdb0d4b6, 9, Finished, Available, Finished, False)

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import lit, desc, row_number, col
from pyspark.sql.window import Window

# Path Setup

abfs_path = 'abfss://house_price_DEV@onelake.dfs.fabric.microsoft.com/houseprice_LH.Lakehouse/Files/Landing'
partition_path = f"/Processing_date={today_date}"
complete_path = abfs_path + partition_path

# Modified schema to preserve original Excel columns
v_schema = StructType([
    StructField("property_id", StringType(), True),
    StructField("month", StringType(), True), 
    StructField("price", DoubleType(), True),
    StructField("area", IntegerType(), True),
    StructField("bedrooms", IntegerType(), True),
    StructField("bathrooms", IntegerType(), True),
    StructField("stories", IntegerType(), True),
    StructField("mainroad", StringType(), True),
    StructField("guestroom", StringType(), True),
    StructField("basement", StringType(), True),
    StructField("hotwaterheating", StringType(), True),
    StructField("airconditioning", StringType(), True),
    StructField("parking", IntegerType(), True),
    StructField("prefarea", StringType(), True),
    StructField("furnishingstatus", StringType(), True)
])

print(f"Loading from: {complete_path}")

StatementMeta(, acb60a16-ca7e-402b-bdd7-8027fdb0d4b6, 10, Finished, Available, Finished, False)

Loading from: abfss://house_price_DEV@onelake.dfs.fabric.microsoft.com/houseprice_LH.Lakehouse/Files/Landing/Processing_date=2026-02-25


In [9]:
# Read the CSV
df = spark.read.format('csv').option('header','True').schema(v_schema).load(complete_path)

# Add processing_date column to the dataframe to match the table structure
df = df.withColumn("processing_date", lit(today_date))

# 3. NEW STEP: Deduplicate to prevent the "Multiple Source Row" error
# We group by the unique keys (id + month) and pick the latest one if duplicates exist
window_spec = Window.partitionBy("property_id", "month").orderBy(desc("processing_date"))

df_deduplicated = (df.withColumn("row_num", row_number().over(window_spec))
                   .filter("row_num = 1")
                   .drop("row_num"))
# Register Temp View
df.createOrReplaceTempView('t_new_data')

# Preview the data
spark.sql("SELECT * FROM t_new_data LIMIT 5").show()

StatementMeta(, acb60a16-ca7e-402b-bdd7-8027fdb0d4b6, 11, Finished, Available, Finished, False)

+-----------+--------------------+-------------+----+--------+---------+-------+--------+---------+--------+---------------+---------------+-------+--------+----------------+---------------+
|property_id|               month|        price|area|bedrooms|bathrooms|stories|mainroad|guestroom|basement|hotwaterheating|airconditioning|parking|prefarea|furnishingstatus|processing_date|
+-----------+--------------------+-------------+----+--------+---------+-------+--------+---------+--------+---------------+---------------+-------+--------+----------------+---------------+
|          0|2023-01-01T00:00:...|1.340120547E7|7420|       4|        2|      3|       1|        0|       0|              0|              1|      2|       1|               2|     2026-02-25|
|          0|2023-02-01T00:00:...|1.347800769E7|7420|       4|        2|      3|       1|        0|       0|              0|              1|      2|       1|               2|     2026-02-25|
|          0|2023-03-01T00:00:...|1.361097296

In [10]:
# Target table path
Fabric_tblsales_bronze = 'abfss://house_price_DEV@onelake.dfs.fabric.microsoft.com/houseprice_LH.Lakehouse/Tables/tblsales_bronze'

try:
    # Try to load and create view
    spark.read.format('delta').load(Fabric_tblsales_bronze).createOrReplaceTempView('t_tblsales_bronze')
    print("Table exists and view created.")
except:
    # Create the table if it doesn't exist
    v_create_table = """
    CREATE TABLE IF NOT EXISTS tblsales_bronze (
        property_id string,
        month string,
        price DOUBLE,
        area int,
        bedrooms int,
        bathrooms int,
        stories int,
        mainroad string,
        guestroom string,
        basement string,
        hotwaterheating string,
        airconditioning string,
        parking int,
        prefarea string,
        furnishingstatus string,
        processing_date string
    ) USING DELTA
    """
    spark.sql(v_create_table)
    print("Table created for the first time.")

StatementMeta(, acb60a16-ca7e-402b-bdd7-8027fdb0d4b6, 12, Finished, Available, Finished, False)

Table created for the first time.


In [11]:
sql_statement = f"""
MERGE INTO tblsales_bronze as target
USING t_new_data as source
ON target.property_id = source.property_id AND target.month = source.month

WHEN MATCHED THEN
    UPDATE SET 
        target.price            = source.price,
        target.area             = source.area,
        target.bedrooms         = source.bedrooms,
        target.bathrooms        = source.bathrooms,
        target.stories          = source.stories,
        target.mainroad         = source.mainroad,
        target.guestroom        = source.guestroom,
        target.basement         = source.basement,
        target.hotwaterheating  = source.hotwaterheating,
        target.airconditioning  = source.airconditioning,
        target.parking          = source.parking,
        target.prefarea         = source.prefarea,
        target.furnishingstatus = source.furnishingstatus,
        target.processing_date  = '{today_date}'

WHEN NOT MATCHED THEN
    INSERT (
        property_id, month, price, area, bedrooms, bathrooms, stories, 
        mainroad, guestroom, basement, hotwaterheating, airconditioning, 
        parking, prefarea, furnishingstatus, processing_date
    )
    VALUES (
        source.property_id, source.month, source.price, source.area, source.bedrooms, 
        source.bathrooms, source.stories, source.mainroad, source.guestroom, 
        source.basement, source.hotwaterheating, source.airconditioning, 
        source.parking, source.prefarea, source.furnishingstatus, '{today_date}'
    )
"""

spark.sql(sql_statement).show()

StatementMeta(, acb60a16-ca7e-402b-bdd7-8027fdb0d4b6, 13, Finished, Available, Finished, False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|            19620|           19620|               0|                0|
+-----------------+----------------+----------------+-----------------+



In [12]:
# Final Check
display(spark.sql("SELECT * FROM tblsales_bronze"))

StatementMeta(, acb60a16-ca7e-402b-bdd7-8027fdb0d4b6, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e4e3d91b-28f7-4c4a-b3b4-9b47da9aa380)

In [13]:
abfs_path='abfss://house_price_DEV@onelake.dfs.fabric.microsoft.com/houseprice_LH.Lakehouse/Files/Landing'
today_date='2026-02-25'
partition_path=f"/Processing_date={today_date}"
complete_path=abfs_path+partition_path
print(complete_path)


StatementMeta(, acb60a16-ca7e-402b-bdd7-8027fdb0d4b6, 15, Finished, Available, Finished, False)

abfss://house_price_DEV@onelake.dfs.fabric.microsoft.com/houseprice_LH.Lakehouse/Files/Landing/Processing_date=2026-02-25
